# 🔍 Post-Mortem Hallucination Detection POC

**Objective:** Evaluate off-the-shelf hallucination detection tools on the
"adjacent-topic, wrong-capability" failure pattern in RAG-based support bots.

**Tools Evaluated:**
1. **LettuceDetect** — Pretrained span-level hallucination classifier (RAGTruth-tuned)
2. **Vectara HHEM-2.1-Open** — Pretrained consistency classifier (Local CPU/GPU via transformers)
3. **Ollama Llama 3.2 (LLM-as-Judge)** — Faithfulness checking via local Ollama API

**Datasets:**
- Custom mock dataset (25 examples modeling the specific failure pattern)
- Synthetic benchmark (sanity check)

---

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = [
    'torch',
    'lettucedetect',
    'sentence-transformers',
    'transformers',
    'sentencepiece',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'requests',
    'accelerate',
]

print('Installing packages...')
for pkg in packages:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f'  \u2705 {pkg}')
    except Exception as e:
        print(f'  \u26a0\ufe0f {pkg} - install issue: {e}')

print('\n\u2705 Dependency installation complete!')


In [ ]:
# ============================================================
# \U0001f527 Imports & System Check
# ============================================================
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import time
import re
import warnings
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, accuracy_score
)
from collections import defaultdict
warnings.filterwarnings('ignore')

# Set visual style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# GPU Check
print('=' * 60)
print('\U0001f5a5\ufe0f  SYSTEM INFORMATION')
print('=' * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cuda_ver = torch.version.cuda
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'\u2705 GPU: {gpu_name}')
    print(f'   CUDA: {cuda_ver}')
    print(f'   VRAM: {vram:.1f} GB')
    DEVICE = 'cuda'
else:
    print('\u26a0\ufe0f  No GPU detected \u2014 using CPU (slower inference)')
    DEVICE = 'cpu'
print(f'   PyTorch: {torch.__version__}')
print(f'   Device: {DEVICE}')
print('=' * 60)


In [ ]:
# ============================================================
# \u2699\ufe0f Configuration
# ============================================================
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3.2:latest'

# Thresholds
HHEM_THRESHOLD = 0.5  # Score > threshold = faithful

# Store results for all tools
all_results = {}

print('Configuration loaded.')
print(f'LLM-as-Judge Model: {OLLAMA_MODEL} (Local via Ollama)')


---
## \U0001f4cb Section 1: Mock Dataset \u2014 "Adjacent-Topic, Wrong-Capability" Pattern


In [ ]:
# ============================================================
# Build Mock Dataset (25 examples)
# ============================================================

mock_dataset = [
    # === HALLUCINATED EXAMPLES (13) ===
    {
        "id": "H01",
        "query": "Can I push/pull code via GitHub integration in Site24x7?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity. "
            "Set up alerts for unusual patterns in your codebase activity. Configure "
            "the integration from Admin > Third Party Integration > GitHub."
        ],
        "response": (
            "Yes, Site24x7 supports full GitHub push/pull integration. You can push "
            "and pull code directly from the Site24x7 dashboard through the GitHub "
            "integration available under Admin > Third Party Integration > GitHub."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H02",
        "query": "Does Zoho CRM support video calling with customers?",
        "retrieved_docs": [
            "Zoho CRM Telephony Integration: Make and receive calls directly within "
            "Zoho CRM using built-in telephony features. Log call details automatically, "
            "record calls for quality assurance, and associate calls with specific leads "
            "or contacts. Supports integration with major telephony providers like Twilio "
            "and RingCentral."
        ],
        "response": (
            "Yes, Zoho CRM supports video calling with customers through its built-in "
            "telephony integration. You can make video calls directly from lead or contact "
            "records using the telephony feature under Settings > Telephony."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H03",
        "query": "Can Zoho Desk automatically translate support tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Zoho Desk supports multiple languages for "
            "your help center portal. Customers can view the help center in their preferred "
            "language. You can create knowledge base articles in multiple languages and "
            "configure language-specific email templates. Supported languages include English, "
            "Spanish, French, German, Japanese, and 40+ others."
        ],
        "response": (
            "Yes, Zoho Desk can automatically translate support tickets into different "
            "languages. When a ticket arrives, the system detects the language and translates "
            "it for your agents. Configure this under Setup > General Settings > Multi-language."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H04",
        "query": "Does Zoho Books support cryptocurrency payments?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in multiple currencies "
            "with automatic exchange rate updates. Zoho Books supports 170+ currencies and "
            "automatically fetches real-time exchange rates. Record transactions, generate "
            "invoices, and manage accounts payable/receivable in foreign currencies."
        ],
        "response": (
            "Yes, Zoho Books supports cryptocurrency payments. You can accept Bitcoin, "
            "Ethereum, and other major cryptocurrencies through the multi-currency payment "
            "gateway. Configure crypto payments under Settings > Online Payments."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H05",
        "query": "Can Zoho Projects automatically assign tasks using AI?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create and organize tasks with customizable "
            "fields, priorities, and deadlines. Assign tasks to team members manually or "
            "in bulk. Use task dependencies to establish project workflows. Track task "
            "progress with Kanban boards, Gantt charts, and custom views."
        ],
        "response": (
            "Yes, Zoho Projects includes AI-powered automatic task assignment. The system "
            "analyzes team members workload, skills, and availability to automatically assign "
            "incoming tasks to the most suitable team member."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H06",
        "query": "Does Zoho Analytics support real-time streaming data ingestion?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from various sources including files "
            "(CSV, Excel, JSON), databases (MySQL, PostgreSQL, SQL Server, Oracle), cloud "
            "storage (Google Drive, Dropbox, Box), and business applications. Schedule "
            "automatic data syncs at intervals ranging from every 30 minutes to daily/weekly."
        ],
        "response": (
            "Yes, Zoho Analytics supports real-time streaming data ingestion. You can set up "
            "live data streams that push data continuously into your analytics workspace. "
            "Configure streaming endpoints under Data Sources > Real-time Streams."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H07",
        "query": "Can Zoho Mail compose AI-generated replies?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses to quickly reply to common "
            "emails with pre-written templates. Create and manage response templates "
            "organized by categories. Set up auto-reply (out-of-office) messages with "
            "customizable schedules. Use email filters and rules to organize incoming mail."
        ],
        "response": (
            "Yes, Zoho Mail can compose AI-generated replies based on email context. The "
            "AI analyzes the incoming email and generates contextual reply suggestions. You "
            "can find this feature under the Smart Reply option when composing a response."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H08",
        "query": "Does Site24x7 support fully automated server self-healing?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automated actions triggered by monitor "
            "alerts. Supported automation actions include restarting Windows services, "
            "executing custom scripts on servers, sending webhook notifications, creating "
            "tickets in ITSM tools, and triggering AWS Lambda functions."
        ],
        "response": (
            "Yes, Site24x7 supports fully automated server self-healing. When issues are "
            "detected, the system automatically diagnoses the root cause and applies the "
            "appropriate fix without human intervention, including automatic scaling, "
            "configuration rollback, and self-healing procedures."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H09",
        "query": "Can Zoho Creator deploy apps to App Store/Play Store?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access your Zoho Creator applications on mobile "
            "devices through the Zoho Creator mobile app. View and manage records, submit "
            "forms, and receive push notifications on iOS and Android. Customize the mobile "
            "layout of your forms and reports for a better mobile experience."
        ],
        "response": (
            "Yes, Zoho Creator allows you to deploy your applications as standalone mobile "
            "apps on the Apple App Store and Google Play Store. You can customize the app "
            "icon, splash screen, and branding via Deploy > App Store Publishing."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H10",
        "query": "Does Zoho Desk support voice-based ticket creation with speech-to-text?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Ticketing: Create tickets from multiple channels "
            "including email, web forms, social media (Twitter, Facebook), live chat, phone "
            "calls, and community forums. Each channel is automatically mapped to a "
            "department. Phone call tickets are created when agents log call details."
        ],
        "response": (
            "Yes, Zoho Desk supports voice-based ticket creation with built-in speech-to-text "
            "AI. Customers can call in and the system automatically transcribes the conversation "
            "in real-time, creating a ticket with the full transcript and auto-detected category."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H11",
        "query": "Can Zoho Books automatically categorize expenses using AI?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Record and manage business expenses. Attach "
            "receipts by uploading images or forwarding email receipts. Create expense "
            "categories and tags for organization. Set up recurring expenses for regular "
            "payments. Generate expense reports by category, vendor, date range, or project."
        ],
        "response": (
            "Yes, Zoho Books uses AI to automatically categorize your expenses. When you "
            "upload a receipt, the AI engine reads the receipt, extracts vendor and amount "
            "information, and automatically assigns the expense to the correct category."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H12",
        "query": "Can Zoho Analytics generate natural language insights from charts?",
        "retrieved_docs": [
            "Zoho Analytics Visualization: Create interactive charts, pivot tables, and "
            "dashboards. Choose from 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets. Apply dynamic filters and drill-down "
            "capabilities. Share dashboards or embed them in websites."
        ],
        "response": (
            "Yes, Zoho Analytics can generate natural language insights from your charts. "
            "The AI-powered narrative engine automatically analyzes your data visualizations "
            "and produces written summaries explaining key trends, outliers, and patterns."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },
    {
        "id": "H13",
        "query": "Does Zoho Projects support automatic time tracking based on app usage?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log time spent on tasks manually or using the "
            "built-in timer. Start/stop the timer from task detail views. Enter time logs "
            "in bulk using the timesheet view. Set budgeted hours for tasks and projects. "
            "Generate timesheet reports by user, project, or date range."
        ],
        "response": (
            "Yes, Zoho Projects includes automatic time tracking that monitors your application "
            "usage. It detects which project-related applications you are using and automatically "
            "logs time to the corresponding tasks via Settings > Time Tracking > Auto Detection."
        ),
        "label": "hallucinated",
        "category": "adjacent-capability"
    },

    # === FAITHFUL EXAMPLES (12) ===
    {
        "id": "F01",
        "query": "Does Site24x7 support GitHub repository monitoring?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity. "
            "Set up alerts for unusual patterns in your codebase activity. Configure "
            "the integration from Admin > Third Party Integration > GitHub."
        ],
        "response": (
            "Yes, Site24x7 supports GitHub repository monitoring. You can track repository "
            "health, commit frequency, pull request metrics, and overall repository activity. "
            "Set up the integration from Admin > Third Party Integration > GitHub."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F02",
        "query": "How do I set up multi-language support in Zoho Desk help center?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Zoho Desk supports multiple languages for "
            "your help center portal. Customers can view the help center in their preferred "
            "language. You can create knowledge base articles in multiple languages and "
            "configure language-specific email templates. Supported languages include English, "
            "Spanish, French, German, Japanese, and 40+ others."
        ],
        "response": (
            "You can set up multi-language support by creating knowledge base articles in "
            "different languages and configuring language-specific email templates. Your help "
            "center portal supports 40+ languages allowing customers to view content in their "
            "preferred language."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F03",
        "query": "Can I handle foreign currency transactions in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in multiple currencies "
            "with automatic exchange rate updates. Zoho Books supports 170+ currencies and "
            "automatically fetches real-time exchange rates. Record transactions, generate "
            "invoices, and manage accounts payable/receivable in foreign currencies."
        ],
        "response": (
            "Yes, Zoho Books supports multi-currency transactions with 170+ currencies. "
            "The system automatically fetches real-time exchange rates. You can record "
            "transactions, generate invoices, and manage accounts in foreign currencies "
            "with automatic conversion."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F04",
        "query": "What task management features are available in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create and organize tasks with customizable "
            "fields, priorities, and deadlines. Assign tasks to team members manually or "
            "in bulk. Use task dependencies to establish project workflows. Track task "
            "progress with Kanban boards, Gantt charts, and custom views."
        ],
        "response": (
            "Zoho Projects offers comprehensive task management including customizable task "
            "fields, priorities, and deadlines. You can assign tasks manually or in bulk, "
            "set up task dependencies for workflows, and track progress using Kanban boards, "
            "Gantt charts, and custom views."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F05",
        "query": "How does Zoho Analytics handle data imports?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from various sources including files "
            "(CSV, Excel, JSON), databases (MySQL, PostgreSQL, SQL Server, Oracle), cloud "
            "storage (Google Drive, Dropbox, Box), and business applications. Schedule "
            "automatic data syncs at intervals ranging from every 30 minutes to daily/weekly."
        ],
        "response": (
            "Zoho Analytics supports importing data from files (CSV, Excel, JSON), databases "
            "(MySQL, PostgreSQL, SQL Server, Oracle), cloud storage services, and business "
            "applications. You can schedule automatic data syncs from every 30 minutes to "
            "weekly intervals."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F06",
        "query": "What are the auto-reply options in Zoho Mail?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses to quickly reply to common "
            "emails with pre-written templates. Create and manage response templates "
            "organized by categories. Set up auto-reply (out-of-office) messages with "
            "customizable schedules. Use email filters and rules to organize incoming mail."
        ],
        "response": (
            "Zoho Mail offers canned responses with pre-written templates organized by "
            "categories for quick replies. You can set up auto-reply out-of-office messages "
            "with customizable schedules. Email filters and rules help automatically organize "
            "incoming mail."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F07",
        "query": "What IT automation actions can I configure in Site24x7?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automated actions triggered by monitor "
            "alerts. Supported automation actions include restarting Windows services, "
            "executing custom scripts on servers, sending webhook notifications, creating "
            "tickets in ITSM tools, and triggering AWS Lambda functions."
        ],
        "response": (
            "Site24x7 supports several IT automation actions triggered by alerts: restarting "
            "Windows services, executing custom server scripts, sending webhook notifications, "
            "creating ITSM tickets, and triggering AWS Lambda functions."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F08",
        "query": "Can I access Zoho Creator apps on my phone?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access your Zoho Creator applications on mobile "
            "devices through the Zoho Creator mobile app. View and manage records, submit "
            "forms, and receive push notifications on iOS and Android. Customize the mobile "
            "layout of your forms and reports for a better mobile experience."
        ],
        "response": (
            "Yes, you can access Zoho Creator apps on mobile through the Zoho Creator mobile "
            "app, available on iOS and Android. You can view and manage records, submit forms, "
            "and receive push notifications. The mobile layout can be customized."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F09",
        "query": "What channels does Zoho Desk support for creating tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Ticketing: Create tickets from multiple channels "
            "including email, web forms, social media (Twitter, Facebook), live chat, phone "
            "calls, and community forums. Each channel is automatically mapped to a "
            "department. Phone call tickets are created when agents log call details."
        ],
        "response": (
            "Zoho Desk supports ticket creation from multiple channels: email, web forms, "
            "social media (Twitter, Facebook), live chat, phone calls, and community forums. "
            "Each channel is automatically mapped to a department."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F10",
        "query": "How do I track expenses in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Record and manage business expenses. Attach "
            "receipts via image upload or email forwarding. Create expense categories and tags "
            "for organization. Set up recurring expenses for regular payments. Generate "
            "expense reports by category, vendor, date range, or project."
        ],
        "response": (
            "You can track expenses in Zoho Books by recording them manually and attaching "
            "receipts via image upload or email forwarding. Organize expenses with categories "
            "and tags, set up recurring expenses, and generate reports by category, vendor, "
            "date, or project."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F11",
        "query": "How do I log time on tasks in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log time spent on tasks manually or using the "
            "built-in timer. Start/stop the timer from task detail views. Enter time logs "
            "in bulk using the timesheet view. Set budgeted hours for tasks and projects. "
            "Generate timesheet reports by user, project, or date range."
        ],
        "response": (
            "You can log time on tasks in Zoho Projects manually or with the built-in timer "
            "from task views. Bulk entry is available via the timesheet view. You can set "
            "budgeted hours and generate timesheet reports by user, project, or date range."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
    {
        "id": "F12",
        "query": "What visualization options does Zoho Analytics offer?",
        "retrieved_docs": [
            "Zoho Analytics Visualization: Create interactive charts, pivot tables, and "
            "dashboards. Choose from 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets. Apply dynamic filters and drill-down "
            "capabilities. Share dashboards or embed them in websites."
        ],
        "response": (
            "Zoho Analytics offers 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets, along with pivot tables and dashboards. "
            "You can apply dynamic filters with drill-down capabilities, share or embed "
            "dashboards."
        ),
        "label": "faithful",
        "category": "correct-answer"
    },
]

import pandas as pd
df_mock = pd.DataFrame(mock_dataset)
print(f"Mock dataset created: {len(mock_dataset)} examples")


---
## \U0001f4da Section 2: Synthetic Benchmark (Sanity Check)


In [ ]:
# ============================================================
# Synthetic Benchmark Dataset (10 clear-cut examples)
# ============================================================

ragtruth_dataset = [
    {
        "id": "SYN01",
        "query": "What is the capital of France?",
        "retrieved_docs": ["France is a country in Western Europe. Its capital city is Paris, known for the Eiffel Tower."],
        "response": "The capital of France is Paris.",
        "label": "faithful",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN02",
        "query": "What is the capital of France?",
        "retrieved_docs": ["France is a country in Western Europe. Its capital city is Paris, known for the Eiffel Tower."],
        "response": "The capital of France is Lyon, the second-largest city.",
        "label": "hallucinated",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN03",
        "query": "How many planets are in our solar system?",
        "retrieved_docs": ["Our solar system has 8 planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune."],
        "response": "There are 8 planets in our solar system.",
        "label": "faithful",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN04",
        "query": "How many planets are in our solar system?",
        "retrieved_docs": ["Our solar system has 8 planets: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune."],
        "response": "There are 12 planets in our solar system, including several recently discovered ones.",
        "label": "hallucinated",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN05",
        "query": "What programming language is used for data science?",
        "retrieved_docs": ["Python is widely used in data science due to libraries like pandas, numpy, and scikit-learn. R is also popular."],
        "response": "Python is widely used for data science, with libraries including pandas, numpy, and scikit-learn. R is also common.",
        "label": "faithful",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN06",
        "query": "What programming language is used for data science?",
        "retrieved_docs": ["Python is widely used in data science due to libraries like pandas, numpy, and scikit-learn. R is also popular."],
        "response": "Java is the primary language for data science, with JavaML being the industry standard framework.",
        "label": "hallucinated",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN07",
        "query": "What does photosynthesis produce?",
        "retrieved_docs": ["Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen."],
        "response": "Photosynthesis produces glucose and oxygen from sunlight, water, and carbon dioxide.",
        "label": "faithful",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN08",
        "query": "What does photosynthesis produce?",
        "retrieved_docs": ["Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen."],
        "response": "Photosynthesis produces proteins and nitrogen, which plants use for cellular structures.",
        "label": "hallucinated",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN09",
        "query": "When was the first moon landing?",
        "retrieved_docs": ["On July 20, 1969, NASA Apollo 11 astronauts Neil Armstrong and Buzz Aldrin became the first humans on the Moon."],
        "response": "The first moon landing was on July 20, 1969, during the Apollo 11 mission.",
        "label": "faithful",
        "category": "synthetic-baseline"
    },
    {
        "id": "SYN10",
        "query": "When was the first moon landing?",
        "retrieved_docs": ["On July 20, 1969, NASA Apollo 11 astronauts Neil Armstrong and Buzz Aldrin became the first humans on the Moon."],
        "response": "The first moon landing was in 1972 during the Apollo 17 mission led by Gene Cernan.",
        "label": "hallucinated",
        "category": "synthetic-baseline"
    },
]

df_ragtruth = pd.DataFrame(ragtruth_dataset)
print(f"Benchmark dataset created: {len(ragtruth_dataset)} examples")


---
## \U0001f96c Section 3: Tool 1 \u2014 LettuceDetect


In [ ]:
# ============================================================
# LettuceDetect Evaluation
# ============================================================

lettuce_results_mock = []
lettuce_results_ragtruth = []

try:
    from lettucedetect.models.inference import HallucinationDetector

    print("Loading LettuceDetect model...")
    detector = HallucinationDetector(
        method="transformer",
        model_path="KRLabsOrg/lettucedect-base-modernbert-en-v1",
    )
    print("\u2705 LettuceDetect loaded!\n")

    # --- Mock dataset ---
    print("Evaluating on Mock Dataset:")
    print("-" * 50)
    for i, row in enumerate(mock_dataset):
        try:
            preds = detector.predict(
                context=row["retrieved_docs"],
                question=row["query"],
                answer=row["response"],
            )
            has_hall = any(p for p in preds) if preds else False
            predicted = "hallucinated" if has_hall else "faithful"
        except Exception as e:
            predicted = "error"
            print(f"  \u26a0\ufe0f Error on {row['id']}: {e}")

        lettuce_results_mock.append(predicted)
        ok = "\u2705" if predicted == row["label"] else "\u274c"
        print(f"  {ok} [{row['id']}] Pred: {predicted:12s} | GT: {row['label']}")

    # --- Benchmark dataset ---
    print("\nEvaluating on Benchmark Dataset:")
    print("-" * 50)
    for i, row in enumerate(ragtruth_dataset):
        try:
            preds = detector.predict(
                context=row["retrieved_docs"],
                question=row["query"],
                answer=row["response"],
            )
            has_hall = any(p for p in preds) if preds else False
            predicted = "hallucinated" if has_hall else "faithful"
        except Exception as e:
            predicted = "error"

        lettuce_results_ragtruth.append(predicted)
        ok = "\u2705" if predicted == row["label"] else "\u274c"
        print(f"  {ok} [{row['id']}] Pred: {predicted:12s} | GT: {row['label']}")

except Exception as e:
    print(f"\u274c LettuceDetect failed: {e}")
    lettuce_results_mock = ["unavailable"] * len(mock_dataset)
    lettuce_results_ragtruth = ["unavailable"] * len(ragtruth_dataset)

all_results["LettuceDetect"] = {
    "mock": lettuce_results_mock,
    "ragtruth": lettuce_results_ragtruth,
}


---
## \U0001f52c Section 4: Tool 2 \u2014 Vectara HHEM-2.1-Open


In [ ]:
# ============================================================
# Vectara HHEM-2.1-Open Evaluation
# ============================================================

hhem_results_mock = []
hhem_results_ragtruth = []
hhem_scores_mock = []
hhem_scores_ragtruth = []

try:
    import sentencepiece
    import transformers
    import torch
    
    # Redefine PreTrainedModel property to include a setter to avoid breaking weights tying loading
    transformers.PreTrainedModel.all_tied_weights_keys = property(
        lambda self: getattr(self, "_all_tied_weights_keys", {}),
        lambda self, v: setattr(self, "_all_tied_weights_keys", v)
    )

    from transformers import AutoModelForSequenceClassification

    print("Loading Vectara HHEM model...")
    model_name = "vectara/hallucination_evaluation_model"
    hhem_model = AutoModelForSequenceClassification.from_pretrained(model_name, trust_remote_code=True).to(DEVICE)
    
    # Copy shared embedding weights to missing encoder embed_tokens weights
    with torch.no_grad():
        hhem_model.t5.transformer.encoder.embed_tokens.weight.data.copy_(hhem_model.t5.transformer.shared.weight.data)
        
    print(f"\u2705 HHEM loaded on {DEVICE}!\n")

    def predict_hhem(premise, hypothesis):
        """Run the model's built-in predict method to get consistency score."""
        scores = hhem_model.predict([(premise, hypothesis)])
        return float(scores[0].item())

    # --- Mock dataset ---
    print("Evaluating on Mock Dataset:")
    print("-" * 50)
    for i, row in enumerate(mock_dataset):
        try:
            docs_text = " ".join(row["retrieved_docs"])
            score = predict_hhem(docs_text, row["response"])
            predicted = "faithful" if score > HHEM_THRESHOLD else "hallucinated"
        except Exception as e:
            score = -1.0
            predicted = "error"
            print(f"  \u26a0\ufe0f Error on {row['id']}: {e}")

        hhem_results_mock.append(predicted)
        hhem_scores_mock.append(score)
        ok = "\u2705" if predicted == row["label"] else "\u274c"
        print(f"  {ok} [{row['id']}] Score: {score:.4f} \u2192 {predicted:12s} | GT: {row['label']}")

    # --- Benchmark dataset ---
    print("\nEvaluating on Benchmark Dataset:")
    print("-" * 50)
    for i, row in enumerate(ragtruth_dataset):
        try:
            docs_text = " ".join(row["retrieved_docs"])
            score = predict_hhem(docs_text, row["response"])
            predicted = "faithful" if score > HHEM_THRESHOLD else "hallucinated"
        except Exception as e:
            score = -1.0
            predicted = "error"

        hhem_results_ragtruth.append(predicted)
        hhem_scores_ragtruth.append(score)
        ok = "\u2705" if predicted == row["label"] else "\u274c"
        print(f"  {ok} [{row['id']}] Score: {score:.4f} \u2192 {predicted:12s} | GT: {row['label']}")

except Exception as e:
    print(f"\u274c HHEM failed: {e}")
    hhem_results_mock = ["unavailable"] * len(mock_dataset)
    hhem_results_ragtruth = ["unavailable"] * len(ragtruth_dataset)
    hhem_scores_mock = [-1.0] * len(mock_dataset)
    hhem_scores_ragtruth = [-1.0] * len(ragtruth_dataset)

all_results["HHEM"] = {
    "mock": hhem_results_mock,
    "ragtruth": hhem_results_ragtruth,
    "scores_mock": hhem_scores_mock,
    "scores_ragtruth": hhem_scores_ragtruth,
}


---
## \U0001f916 Section 5: Tool 3 \u2014 Local LLM-as-Judge (Ollama Llama 3.2)


In [ ]:
# ============================================================
# Local Ollama LLM-as-Judge Evaluation
# ============================================================

def ollama_faithfulness_check(query, docs, response):
    """Check faithfulness using local Ollama model."""
    docs_text = "\n\n".join(docs)

    prompt = (
        "You are a strict hallucination detection system for a support bot.\n\n"
        "TASK: Determine if the Response is FAITHFUL or HALLUCINATED with respect to ONLY the Source Documents.\n\n"
        "CRITICAL RULES:\n"
        "1. Response is HALLUCINATED if it claims a feature/capability NOT explicitly mentioned in the source documents.\n"
        "2. Being topically related is NOT enough. The specific capability must be stated.\n"
        "3. If docs describe Feature A (monitoring) but response claims Feature B (push/pull), that is HALLUCINATED.\n"
        "4. Response is FAITHFUL only if every factual claim is directly supported by the documents.\n\n"
        f"SOURCE DOCUMENTS:\n{docs_text}\n\n"
        f"USER QUERY:\n{query}\n\n"
        f"RESPONSE TO EVALUATE:\n{response}\n\n"
        'Respond ONLY with JSON format: {"verdict": "faithful" or "hallucinated", "confidence": 0.0-1.0, "reason": "one sentence explanation"}'
    )

    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {
                    "temperature": 0.0
                }
            },
            timeout=30,
        )

        if resp.status_code != 200:
            return {"verdict": "error", "confidence": 0.0, "reason": f"Ollama HTTP {resp.status_code}"}

        result = resp.json()
        text = result.get("response", "").strip()
        parsed = json.loads(text)
        return parsed

    except Exception as e:
        return {"verdict": "error", "confidence": 0.0, "reason": str(e)[:200]}


ollama_results_mock = []
ollama_results_ragtruth = []
ollama_details_mock = []
ollama_details_ragtruth = []

print(f"Evaluating with Local Ollama ({OLLAMA_MODEL})...\n")
print("Mock Dataset:")
print("-" * 50)
for i, row in enumerate(mock_dataset):
    result = ollama_faithfulness_check(row["query"], row["retrieved_docs"], row["response"])
    predicted = result.get("verdict", "error")
    confidence = result.get("confidence", 0.0)
    reason = result.get("reason", "N/A")

    ollama_results_mock.append(predicted)
    ollama_details_mock.append(result)

    ok = "\u2705" if predicted == row["label"] else "\u274c"
    print(f"  {ok} [{row['id']}] {predicted:12s} (conf={confidence:.2f}) | GT: {row['label']}")
    print(f"           Reason: {reason[:90]}")

print("\nBenchmark Dataset:")
print("-" * 50)
for i, row in enumerate(ragtruth_dataset):
    result = ollama_faithfulness_check(row["query"], row["retrieved_docs"], row["response"])
    predicted = result.get("verdict", "error")
    confidence = result.get("confidence", 0.0)

    ollama_results_ragtruth.append(predicted)
    ollama_details_ragtruth.append(result)

    ok = "\u2705" if predicted == row["label"] else "\u274c"
    print(f"  {ok} [{row['id']}] {predicted:12s} (conf={confidence:.2f}) | GT: {row['label']}")

all_results["Local-LLM"] = {
    "mock": ollama_results_mock,
    "ragtruth": ollama_results_ragtruth,
    "details_mock": ollama_details_mock,
    "details_ragtruth": ollama_details_ragtruth,
}


---
## \U0001f4ca Section 6: Metrics Computation


In [ ]:
# ============================================================
# Compute Metrics
# ============================================================

def compute_metrics(y_true, y_pred, tool_name, dataset_name):
    """Compute classification metrics (hallucinated = positive class)."""
    valid_mask = [(p not in ("error", "unavailable")) for p in y_pred]
    y_true_f = [t for t, v in zip(y_true, valid_mask) if v]
    y_pred_f = [p for p, v in zip(y_pred, valid_mask) if v]

    if not y_true_f:
        return {"tool": tool_name, "dataset": dataset_name,
                "precision": None, "recall": None, "f1": None, "accuracy": None,
                "n_valid": 0, "n_total": len(y_true)}

    y_true_bin = [1 if t == "hallucinated" else 0 for t in y_true_f]
    y_pred_bin = [1 if p == "hallucinated" else 0 for p in y_pred_f]

    return {
        "tool": tool_name,
        "dataset": dataset_name,
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall":    recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "f1":        f1_score(y_true_bin, y_pred_bin, zero_division=0),
        "accuracy":  accuracy_score(y_true_bin, y_pred_bin),
        "n_valid":   len(y_true_f),
        "n_total":   len(y_true),
    }

gt_mock     = [x["label"] for x in mock_dataset]
gt_ragtruth = [x["label"] for x in ragtruth_dataset]

metrics_rows = []
for tool_name in ["LettuceDetect", "HHEM", "Local-LLM"]:
    data = all_results.get(tool_name, {})
    if data.get("mock"):
        metrics_rows.append(compute_metrics(gt_mock, data["mock"], tool_name, "Mock (Adjacent-Cap.)"))
    if data.get("ragtruth"):
        metrics_rows.append(compute_metrics(gt_ragtruth, data["ragtruth"], tool_name, "Benchmark"))

df_metrics = pd.DataFrame(metrics_rows)

print("=" * 85)
print("📊  HALLUCINATION DETECTION RESULTS")
print("=" * 85)
print()

display_df = df_metrics.copy()
for col in ["precision", "recall", "f1", "accuracy"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if x is not None else "N/A")
display_df["coverage"] = display_df.apply(lambda r: f"{r['n_valid']}/{r['n_total']}", axis=1)

print(display_df[["tool", "dataset", "precision", "recall", "f1", "accuracy", "coverage"]].to_string(index=False))


---
## \U0001f4c8 Section 7: Visualizations


In [ ]:
# ============================================================
# Comparative Visualizations
# ============================================================

valid_metrics = df_metrics[df_metrics["precision"].notna()].copy()

if valid_metrics.empty:
    print("⚠️ No valid metrics to visualize.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Hallucination Detection Tool Comparison", fontsize=16, fontweight="bold", y=1.02)

    # --- Plot 1: P/R/F1 on Mock ---
    ax1 = axes[0, 0]
    mock_v = valid_metrics[valid_metrics["dataset"] == "Mock (Adjacent-Cap.)"]
    if not mock_v.empty:
        x = np.arange(len(mock_v))
        w = 0.25
        b1 = ax1.bar(x - w, mock_v["precision"], w, label="Precision", color="#2196F3", alpha=0.85)
        b2 = ax1.bar(x,     mock_v["recall"],    w, label="Recall",    color="#FF9800", alpha=0.85)
        b3 = ax1.bar(x + w, mock_v["f1"],        w, label="F1",        color="#4CAF50", alpha=0.85)
        ax1.set_xticks(x)
        ax1.set_xticklabels(mock_v["tool"].values, rotation=0)
        ax1.set_ylabel("Score")
        ax1.set_title("Mock Dataset (Adjacent-Capability)", fontweight="bold")
        ax1.set_ylim(0, 1.15)
        ax1.legend(loc="upper right")
        for bars in [b1, b2, b3]:
            for bar in bars:
                h = bar.get_height()
                ax1.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                             xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

    # --- Plot 2: P/R/F1 on Benchmark ---
    ax2 = axes[0, 1]
    bench_v = valid_metrics[valid_metrics["dataset"] == "Benchmark"]
    if not bench_v.empty:
        x = np.arange(len(bench_v))
        w = 0.25
        b1 = ax2.bar(x - w, bench_v["precision"], w, label="Precision", color="#2196F3", alpha=0.85)
        b2 = ax2.bar(x,     bench_v["recall"],    w, label="Recall",    color="#FF9800", alpha=0.85)
        b3 = ax2.bar(x + w, bench_v["f1"],        w, label="F1",        color="#4CAF50", alpha=0.85)
        ax2.set_xticks(x)
        ax2.set_xticklabels(bench_v["tool"].values, rotation=0)
        ax2.set_ylabel("Score")
        ax2.set_title("Benchmark Dataset", fontweight="bold")
        ax2.set_ylim(0, 1.15)
        ax2.legend(loc="upper right")
        for bars in [b1, b2, b3]:
            for bar in bars:
                h = bar.get_height()
                ax2.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                             xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

    # --- Plot 3: Accuracy comparison ---
    ax3 = axes[1, 0]
    pivot = valid_metrics.pivot(index="tool", columns="dataset", values="accuracy")
    pivot.plot(kind="bar", ax=ax3, alpha=0.85, rot=0)
    ax3.set_ylabel("Accuracy")
    ax3.set_title("Overall Accuracy by Dataset", fontweight="bold")
    ax3.set_ylim(0, 1.15)
    ax3.legend(title="Dataset")

    # --- Plot 4: HHEM Score Distribution ---
    ax4 = axes[1, 1]
    scores_m = all_results.get("HHEM", {}).get("scores_mock", [])
    valid_sc = [(s, gt_mock[i]) for i, s in enumerate(scores_m) if s >= 0]
    if valid_sc:
        h_sc = [s for s, l in valid_sc if l == "hallucinated"]
        f_sc = [s for s, l in valid_sc if l == "faithful"]
        if h_sc:
            ax4.hist(h_sc, bins=12, alpha=0.7, label="Hallucinated (GT)", color="#F44336")
        if f_sc:
            ax4.hist(f_sc, bins=12, alpha=0.7, label="Faithful (GT)", color="#4CAF50")
        ax4.axvline(x=HHEM_THRESHOLD, color="black", linestyle="--", linewidth=2,
                    label=f"Threshold ({HHEM_THRESHOLD})")
        ax4.set_xlabel("HHEM Score")
        ax4.set_ylabel("Count")
        ax4.set_title("HHEM Score Distribution (Mock)", fontweight="bold")
        ax4.legend()

    plt.tight_layout()
    plt.savefig("hallucination_detection_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Chart saved to hallucination_detection_results.png")


In [ ]:
# ============================================================
# Confusion Matrices
# ============================================================

tools_ok = []
for tn in ["LettuceDetect", "HHEM", "Local-LLM"]:
    preds = all_results.get(tn, {}).get("mock", [])
    if preds and not any(p in ("error", "unavailable") for p in preds):
        tools_ok.append(tn)

if tools_ok:
    n = len(tools_ok)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, tn in zip(axes, tools_ok):
        preds = all_results[tn]["mock"]
        y_t = [1 if gt_mock[i] == "hallucinated" else 0 for i in range(len(preds))]
        y_p = [1 if preds[i] == "hallucinated" else 0 for i in range(len(preds))]
        cm = confusion_matrix(y_t, y_p, labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=["Faithful", "Hallucinated"],
                    yticklabels=["Faithful", "Hallucinated"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(f"{tn}\n(Mock Dataset)", fontweight="bold")

    plt.suptitle("Confusion Matrices — Mock Dataset", fontsize=14, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
    plt.show()


---
## \U0001f4dd Section 8: Summary & Recommendation


In [ ]:
# ============================================================
# FINAL SUMMARY & RECOMMENDATION
# ============================================================

print("=" * 80)
print("📝 HALLUCINATION DETECTION POC — FINAL SUMMARY")
print("=" * 80)
print()

header = f"{'Tool':17s} {'Dataset':28s} {'Prec':>9s} {'Recall':>8s} {'F1':>8s} {'Acc':>8s}"
print(header)
print("-" * len(header))
for _, row in df_metrics.iterrows():
    p = f"{row['precision']:.3f}" if row['precision'] is not None else "  N/A"
    r = f"{row['recall']:.3f}" if row['recall'] is not None else "  N/A"
    f = f"{row['f1']:.3f}" if row['f1'] is not None else "  N/A"
    a = f"{row['accuracy']:.3f}" if row['accuracy'] is not None else "  N/A"
    print(f"{row['tool']:17s} {row['dataset']:28s} {p:>9s} {r:>8s} {f:>8s} {a:>8s}")

# Recommend based on Local-LLM and HHEM
mock_m = df_metrics[df_metrics["dataset"] == "Mock (Adjacent-Cap.)"]
valid_mock = mock_m[mock_m["f1"].notna()]

if not valid_mock.empty:
    best = valid_mock.loc[valid_mock["f1"].idxmax()]
    print(f"\n🏆 Best on adjacent-capability cases: {best['tool']} (F1={best['f1']:.3f})")
